# Predicting and Inferring Climate Metrics in the United States:
# A 2016 Weekly Analysis of Temperature and Precipitation

This study uses weather data from the National Weather Service (NWS), a division of the National Oceanic and Atmospheric Administration (NOAA), which collects daily weather observations through Weather Forecast Offices across the United States. The data, provided by the CORGIS Dataset Project, summarizes these observations at a weekly level for cities nationwide throughout 2016.

Using this dataset, I examine seasonal patterns in temperature and precipitation, explore relationships between different weather variables, build predictive models for average weekly temperature, and perform inferential analyses to compare climate differences between regions.합니다.

## Problem Formulation

### Descriptive Question
Weather conditions in the United States tend to follow clear seasonal patterns throughout the year. In this section, I start by examining monthly averages of temperature and precipitation for 2016 in order to understand how these variables evolve over time. Summarizing the data at the monthly level allows me to focus on broad trends rather than short-term fluctuations, helping to establish an overall picture of national climate behavior that will guide the analyses that follow.

### Exploratory Question
While temperature and precipitation are often assumed to be related, the nature of this relationship is not always obvious. To explore whether such a relationship exists in the data, a scatter plot was created with temperature and precipitation as the primary variables.

### Predictive Question
To estimate average weekly temperature, a linear regression model was constructed using month, wind speed, and precipitation as explanatory variables. These predictors were selected because they capture temporal effects as well as atmospheric conditions that may influence temperature.

### Inferential Question
Regional differences in climate can be substantial, but observed differences in sample data may arise from random variation. To assess whether California and New York differ meaningfully in terms of mean temperature, a bootstrap resampling procedure was applied to estimate the distribution of the difference in means. The resulting confidence interval provides a basis for determining whether the observed difference reflects a statistically significant population-level effect. This inferential approach emphasizes uncertainty and supports evidence-based comparison between regions.

In [14]:
import pandas as pd
import numpy as np

weather = pd.read_csv('Weather.csv')

weather.columns = [c.split('.')[-1].strip() for c in weather.columns]

weather_tidy = weather[['Avg Temp', 'Precipitation', 'Speed', 'Month', 'State', 'Year']].dropna()

weather_tidy = weather_tidy.rename(columns={
    'Avg Temp': 'Avg_Temp',
    'Speed': 'Wind_Speed'
})

weather_tidy = weather_tidy[weather_tidy['Year'] == 2016]

def get_season(month):
    if month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    elif month in [9, 10, 11]: return 'Fall'
    else: return 'Winter'

weather_tidy['Season'] = weather_tidy['Month'].apply(get_season)

print(weather_tidy.head())

   Avg_Temp  Precipitation  Wind_Speed  Month    State  Year  Season
0        39           0.00        4.33      1  Alabama  2016  Winter
1        39           0.00        3.86      1  Alabama  2016  Winter
2        46           0.16        9.73      1  Alabama  2016  Winter
3        45           0.00        6.86      1  Alabama  2016  Winter
4        34           0.01        7.80      1   Alaska  2016  Winter


This code cleans and prepares the weather dataset for analysis by selecting key variables, renaming columns for clarity, and removing missing values. The data is then filtered to include only observations from 2016, and a new season variable is created based on the month. The resulting tidy dataset is ready for use in descriptive, exploratory, predictive, and inferential analyses.

In [15]:
import altair as alt

monthly_summary = weather_tidy.groupby('Month')[['Avg_Temp', 'Precipitation']].mean().reset_index()

temp_line = alt.Chart(monthly_summary, title="Monthly Average Temperature in 2016").mark_line(point=True).encode(
    x=alt.X("Month:O").title("Month"),
    y=alt.Y("Avg_Temp:Q").scale(zero=False).title("Average Temperature (F)")
).properties(width=500, height=300)

precip_bar = alt.Chart(monthly_summary, title="Monthly Average Precipitation in 2016").mark_bar().encode(
    x=alt.X("Month:O").title("Month"),
    y=alt.Y("Precipitation:Q").title("Precipitation (Inches)")
).properties(width=500, height=300)

combined_plot = alt.vconcat(temp_line, precip_bar).configure_axis(titleFontSize=12)
combined_plot.display()

alt.VConcatChart(...)